In [1]:
import os

from dotenv import load_dotenv

load_dotenv()  # reads course/.env
os.environ.setdefault("DEEPEVAL_TELEMETRY_OPT_OUT", "YES")

from deepeval.models import OpenAIModel

judge = OpenAIModel(
    model="gpt-4.1",
    api_key=os.environ["OPENAI_API_KEY"],
    temperature=0,
)

In [2]:
day15_scores = {
    "Contextual Precision": 0.67,
    "Contextual Recall": 0.67,
    "Contextual Relevancy": 0.28,
    "Faithfulness": 1.00,
    "Answer Relevancy": 0.67,
}

for metric, score in day15_scores.items():
    print(f"{metric:<25} {score:.2f}")

Contextual Precision      0.67
Contextual Recall         0.67
Contextual Relevancy      0.28
Faithfulness              1.00
Answer Relevancy          0.67


In [3]:
CORPUS = [
    {
        "title": "Tokens",
        "content": (
            "Large language models split text into tokens, common character sequences roughly 4 "
            "characters or three-quarters of a word long. Both the prompt and the output are counted "
            "in tokens, and pricing and context limits are measured in tokens."
        ),
    },
    {
        "title": "Embeddings",
        "content": (
            "An embedding is a fixed-length vector that represents the meaning of a piece of text. "
            "Texts with similar meaning have vectors that are close together, usually measured by "
            "cosine similarity, which is what lets a system do semantic search."
        ),
    },
    {
        "title": "Retrieval-Augmented Generation",
        "content": (
            "RAG grounds an LLM's answer in external documents. At query time the system retrieves "
            "the most relevant chunks and passes them to the model as context, which reduces "
            "hallucination and lets you update knowledge without retraining."
        ),
    },
    {
        "title": "Hallucination",
        "content": (
            "A hallucination is fluent, confident text that is factually wrong or unsupported by "
            "its sources. It happens because the model predicts likely text rather than looking "
            "facts up. Grounding answers in retrieved context is the main defense."
        ),
    },
    {
        "title": "AI agents",
        "content": (
            "An AI agent is an LLM given a goal, tools, and a loop: plan, call a tool, observe the "
            "result, decide the next step, until the task is done."
        ),
    },
]

In [ ]:
# This was our retriever from Day 11.
def retrieve_keyword(query: str, k: int = 2) -> list[str]:
    """Rank documents by simple keyword overlap."""

    q_words = set(query.lower().split())

    scored = []

    for doc in CORPUS:
        doc_words = set(
            (doc["title"] + " " + doc["content"])
            .lower()
            .split()
        )

        overlap = len(q_words & doc_words)

        scored.append((overlap, doc))

    scored.sort(
        key=lambda x: x[0],
        reverse=True,
    )

    return [
        doc["content"]
        for _, doc in scored[:k]
    ]

In [ ]:
query = "What is a token in an LLM?"

results = retrieve_keyword(query, k=3)

for i, passage in enumerate(results, 1):
    print(f"{i}. {passage[:100]}...")


# The problem is that this retriever relies heavily on literal word overlap.

1. An embedding is a fixed-length vector that represents the meaning of a piece of text. Texts with sim...
2. A hallucination is fluent, confident text that is factually wrong or unsupported by its sources. It ...
3. An AI agent is an LLM given a goal, tools, and a loop: plan, call a tool, observe the result, decide...


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# Convert our documents into TF-IDF vectors:
docs = [
    f"{doc['title']}. {doc['content']}"
    for doc in CORPUS
]

vectorizer = TfidfVectorizer(
    stop_words="english"
)

doc_vectors = vectorizer.fit_transform(docs)

In [8]:
def retrieve_tfidf(
    query: str,
    k: int = 2,
) -> list[str]:

    query_vector = vectorizer.transform([query])

    similarities = cosine_similarity(
        query_vector,
        doc_vectors,
    )[0]

    ranked = sorted(
        zip(similarities, docs),
        key=lambda x: x[0],
        reverse=True,
    )

    return [
        doc
        for _, doc in ranked[:k]
    ]

In [9]:
for query in [
    "What is a token in an LLM?",
    "How does RAG reduce hallucination?",
]:

    print("\nQ:", query)

    passages = retrieve_tfidf(
        query,
        k=2,
    )

    for i, passage in enumerate(passages, 1):
        print(f"{i}. {passage[:100]}...")


Q: What is a token in an LLM?
1. AI agents. An AI agent is an LLM given a goal, tools, and a loop: plan, call a tool, observe the res...
2. Retrieval-Augmented Generation. RAG grounds an LLM's answer in external documents. At query time the...

Q: How does RAG reduce hallucination?
1. Retrieval-Augmented Generation. RAG grounds an LLM's answer in external documents. At query time the...
2. Hallucination. A hallucination is fluent, confident text that is factually wrong or unsupported by i...


In [ ]:
# Compare the old and new retrievers
query = "What is a token in an LLM?"

print("=== KEYWORD RETRIEVER ===")

for i, passage in enumerate(
    retrieve_keyword(query, k=2),
    1,
):
    print(f"{i}. {passage[:120]}...")


print("\n=== TF-IDF RETRIEVER ===")

for i, passage in enumerate(
    retrieve_tfidf(query, k=2),
    1,
):
    print(f"{i}. {passage[:120]}...")

=== KEYWORD RETRIEVER ===
1. An embedding is a fixed-length vector that represents the meaning of a piece of text. Texts with similar meaning have ve...
2. A hallucination is fluent, confident text that is factually wrong or unsupported by its sources. It happens because the ...

=== TF-IDF RETRIEVER ===
1. AI agents. An AI agent is an LLM given a goal, tools, and a loop: plan, call a tool, observe the result, decide the next...
2. Retrieval-Augmented Generation. RAG grounds an LLM's answer in external documents. At query time the system retrieves th...


In [ ]:
# To make this a fair experiment, we change only the retriever.
from openai import OpenAI

client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"]
)

RAG_SYSTEM = (
    "Answer ONLY using the provided context. "
    "If the context doesn't contain the answer, say you "
    "don't have that information. "
    "Be concise (1-2 sentences)."
)

In [12]:
def answer_with_openai(
    question: str,
    passages: list[str],
) -> str:

    context = "\n\n".join(passages)

    prompt = (
        f"Context:\n{context}\n\n"
        f"Question: {question}\n\n"
        "Answer:"
    )

    response = client.chat.completions.create(
        model="gpt-4.1",
        messages=[
            {
                "role": "system",
                "content": RAG_SYSTEM,
            },
            {
                "role": "user",
                "content": prompt,
            },
        ],
        temperature=0,
    )

    return response.choices[0].message.content.strip()

In [ ]:
# Build the improved RAG pipeline
def ask_rag(
    question: str,
    k: int = 2,
) -> tuple[str, list[str]]:

    passages = retrieve_tfidf(
        question,
        k=k,
    )

    answer = answer_with_openai(
        question,
        passages,
    )

    return answer, passages

In [ ]:
# The test questions must remain unchanged from Day 15.
QUESTIONS = [
    {
        "input": "What is a token in an LLM?",
        "expected_output": (
            "A token is a common character sequence, roughly 4 characters, "
            "used to measure both prompt and output length."
        ),
    },
    {
        "input": "How does RAG reduce hallucination?",
        "expected_output": (
            "RAG retrieves relevant chunks and passes them as context, "
            "grounding the answer in real documents."
        ),
    },
    {
        "input": "How do I containerize a model for deployment with Docker?",
    },
]

In [ ]:
# Run the pipeline:

rows = []

for q in QUESTIONS:

    answer, passages = ask_rag(
        q["input"]
    )

    rows.append(
        {
            **q,
            "actual_output": answer,
            "retrieval_context": passages,
        }
    )

    print("Q:", q["input"])
    print("A:", answer)
    print("Retrieved:", len(passages))
    print("-" * 80)

Q: What is a token in an LLM?
A: The context does not provide information about what a token is in an LLM.
Retrieved: 2
--------------------------------------------------------------------------------
Q: How does RAG reduce hallucination?
A: RAG reduces hallucination by grounding the LLM's answers in external documents retrieved at query time, ensuring responses are supported by relevant context rather than generated solely from the model's predictions.
Retrieved: 2
--------------------------------------------------------------------------------
Q: How do I containerize a model for deployment with Docker?
A: I don't have that information.
Retrieved: 2
--------------------------------------------------------------------------------


In [16]:
from deepeval import evaluate
from deepeval.evaluate.configs import (
    AsyncConfig,
    ErrorConfig,
)

from deepeval.metrics import (
    AnswerRelevancyMetric,
    ContextualPrecisionMetric,
    ContextualRecallMetric,
    ContextualRelevancyMetric,
    FaithfulnessMetric,
)

from deepeval.test_case import LLMTestCase

In [17]:
test_cases = [
    LLMTestCase(
        input=row["input"],
        actual_output=row["actual_output"],
        expected_output=row.get("expected_output"),
        retrieval_context=row["retrieval_context"],
    )
    for row in rows
]

In [18]:
metrics = [
    FaithfulnessMetric(
        model=judge,
        threshold=0.5,
    ),
    AnswerRelevancyMetric(
        model=judge,
        threshold=0.5,
    ),
    ContextualRelevancyMetric(
        model=judge,
        threshold=0.5,
    ),
    ContextualPrecisionMetric(
        model=judge,
        threshold=0.5,
    ),
    ContextualRecallMetric(
        model=judge,
        threshold=0.5,
    ),
]

In [19]:
results = evaluate(
    test_cases=test_cases,
    metrics=metrics,
    async_config=AsyncConfig(
        max_concurrent=2,
        throttle_value=2.0,
    ),
    error_config=ErrorConfig(
        ignore_errors=True,
        skip_on_missing_params=True,
    ),
)

✨ You're running DeepEval's latest Faithfulness Metric! (using gpt-4.1, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-4.1, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Contextual Relevancy Metric! (using gpt-4.1, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Contextual Precision Metric! (using gpt-4.1, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Contextual Recall Metric! (using gpt-4.1, strict=False, async_mode=True)...

c:\Users\T14s\AppData\Local\Programs\Python\Python311\Lib\site-packages\rich\live.py:231: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_0                                                                                                 │
│  ├──   Input:              What is a token in an LLM?                                                           │
│  │     Actual Output:      The context does not provide information about what a token is in an LLM.            │
│  │     Expected Output:    A token is a common character sequence, roughly 4 characters, used to measure        │
│  │                         both prompt and output length.                                                       │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric               ┃ Score ┃ Threshold ┃ Reason                                                │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        PASS  │ Faithfulness         │ 1.00  │ 0.50      │ The score is 1.00 because there are no contradi...    │
│        FAIL  │ Answer Relevancy     │ 0.00  │ 0.50      │ The score is 0.00 because the response did not        │
│              │                      │       │           │ answer the question about what a token is in an LLM   │
│              │                      │       │           │ and instead only commented on the lack of             │
│              │                      │       │           │ information, making it completely irrelevant to the   │
│              │                      │       │           │ input.                                                │
│        FAIL  │ Contextual Relevancy │ 0.00  │ 0.50      │ The score is 0.00 because none of the retrieval       │
│              │                      │       │           │ context statements mention or explain what a token    │
│              │                      │       │           │ is in an LLM, as highlighted by the irrelevancy       │
│              │                      │       │           │ reasons and the absence of any relevant statements.   │
│        FAIL  │ Contextual Precision │ 0.00  │ 0.50      │ The score is 0.00 because both the first and second   │
│              │                      │       │           │ nodes in the retrieval contexts are irrelevant to     │
│              │                      │       │           │ the question. The first node, ranked 1, only          │
│              │                      │       │           │ discusses AI agents and their operation, stating it   │
│              │                      │       │           │ 'does not mention tokens or their definition in the   │
│              │                      │       │           │ context of LLMs.' The second node, ranked 2,          │
│              │                      │       │           │ explains RAG and its use with LLMs but 'does not      │
│              │                      │       │           │ address what a token is or provide any relevant       │
│              │                      │       │           │ information about tokens.' Since all top-ranked       │
│              │                      │       │           │ nodes are irrelevant, the score cannot be higher.     │
│        FAIL  │ Contextual Recall    │ 0.00  │ 0.50      │ The score is 0.00 because none of the nodes in the    │
│              │                      │       │           

⚠ WARNING: No hyperparameters logged.
» ]8;id=474146;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 11.39s | token cost: 0.057048 USD)
» Test Results (3 total tests):
   » Pass Rate: 33.33% | Passed: 1 | Failed: 2

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [20]:
for i, result in enumerate(
    results.test_results,
    start=1,
):

    print(f"\n================ CASE {i} ================")
    print(f"Question: {result.input}")

    for metric_result in result.metrics_data:

        print(
            f"{metric_result.name:<25} "
            f"score={metric_result.score:.2f} "
            f"success={metric_result.success}"
        )


================ CASE 1 ================
Question: What is a token in an LLM?
Faithfulness              score=1.00 success=True
Answer Relevancy          score=0.00 success=False
Contextual Relevancy      score=0.00 success=False
Contextual Precision      score=0.00 success=False
Contextual Recall         score=0.00 success=False

================ CASE 2 ================
Question: How does RAG reduce hallucination?
Faithfulness              score=1.00 success=True
Answer Relevancy          score=1.00 success=True
Contextual Relevancy      score=1.00 success=True
Contextual Precision      score=1.00 success=True
Contextual Recall         score=1.00 success=True

================ CASE 3 ================
Question: How do I containerize a model for deployment with Docker?
Faithfulness              score=1.00 success=True
Answer Relevancy          score=0.00 success=False
Contextual Relevancy      score=0.00 success=False


In [21]:
new_scores = {}

for result in results.test_results:
    for metric_result in result.metrics_data:

        metric_name = metric_result.name

        new_scores.setdefault(
            metric_name,
            [],
        ).append(metric_result.score)


print("=" * 70)
print("DAY 15 vs DAY 16")
print("=" * 70)

for metric_name, old_score in day15_scores.items():

    scores = new_scores.get(metric_name, [])

    if not scores:
        continue

    new_score = sum(scores) / len(scores)

    change = new_score - old_score

    print(
        f"{metric_name:<25} "
        f"Day 15={old_score:.2f}  "
        f"Day 16={new_score:.2f}  "
        f"Change={change:+.2f}"
    )

DAY 15 vs DAY 16
Contextual Precision      Day 15=0.67  Day 16=0.50  Change=-0.17
Contextual Recall         Day 15=0.67  Day 16=0.50  Change=-0.17
Contextual Relevancy      Day 15=0.28  Day 16=0.33  Change=+0.05
Faithfulness              Day 15=1.00  Day 16=1.00  Change=+0.00
Answer Relevancy          Day 15=0.67  Day 16=0.33  Change=-0.34


In [22]:
print(
    """
Engineering decision:

We changed only the retriever.

The evaluation dataset and evaluation metrics
remained unchanged.

Now compare the Day 15 and Day 16 scores.

If retrieval metrics improved, the new retriever
is a better candidate.

If they did not improve, we should investigate
the failure rather than assuming the new retriever
is better.
"""
)


Engineering decision:

We changed only the retriever.

The evaluation dataset and evaluation metrics
remained unchanged.

Now compare the Day 15 and Day 16 scores.

If retrieval metrics improved, the new retriever
is a better candidate.

If they did not improve, we should investigate
the failure rather than assuming the new retriever
is better.

